In [3]:
import pandas as pd

In [4]:
df=pd.read_csv("../data/raw/college_student_placement.csv")
df.head()

,College_ID,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Internship_Experience,Extra_Curricular_Score,Communication_Skills,Projects_Completed,Placement
0,CLG0030,107,6.61,6.28,8,No,8,8,4,No
1,CLG0061,97,5.52,5.37,8,No,7,8,0,No
2,CLG0036,109,5.36,5.83,9,No,3,1,1,No
3,CLG0055,122,5.47,5.75,6,Yes,1,6,1,No
4,CLG0004,96,7.91,7.69,7,No,8,10,2,No


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   College_ID              10000 non-null  str    
 1   IQ                      10000 non-null  int64  
 2   Prev_Sem_Result         10000 non-null  float64
 3   CGPA                    10000 non-null  float64
 4   Academic_Performance    10000 non-null  int64  
 5   Internship_Experience   10000 non-null  str    
 6   Extra_Curricular_Score  10000 non-null  int64  
 7   Communication_Skills    10000 non-null  int64  
 8   Projects_Completed      10000 non-null  int64  
 9   Placement               10000 non-null  str    
dtypes: float64(2), int64(5), str(3)
memory usage: 894.3 KB


In [6]:
print("INTENSHIP:",df["Internship_Experience"].unique())

INTENSHIP: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [7]:
print("PLACEMENT:",df["Placement"].unique())

PLACEMENT: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [8]:
print(df["Placement"].value_counts())

Placement
No     8341
Yes    1659
Name: count, dtype: int64


In [9]:
df.describe()

,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Extra_Curricular_Score,Communication_Skills,Projects_Completed
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,99.471800,7.535673,7.532379,5.546400,4.970900,5.561800,2.513400
std,15.053101,1.447519,1.470141,2.873477,3.160103,2.900866,1.715959
min,41.000000,5.000000,4.540000,1.000000,0.000000,1.000000,0.000000
25%,89.000000,6.290000,6.290000,3.000000,2.000000,3.000000,1.000000
50%,99.000000,7.560000,7.550000,6.000000,5.000000,6.000000,3.000000
75%,110.000000,8.790000,8.770000,8.000000,8.000000,8.000000,4.000000
max,158.000000,10.000000,10.460000,10.000000,10.000000,10.000000,5.000000


In [10]:
df.groupby("Placement")[["CGPA","IQ","Projects_Completed","Communication_Skills"]].mean()

,CGPA,IQ,Projects_Completed,Communication_Skills
Placement,,,,
No,7.321527,97.552452,2.346961,5.143748
Yes,8.592483,109.121760,3.350211,7.663653


In [11]:
df['Placement']=df['Placement'].map({"Yes":1,"No":0})
df["Internship_Experience"]=df["Internship_Experience"].map({"Yes":1,"No":0})
df=df.drop(columns=["College_ID"])
df.head()

,IQ,Prev_Sem_Result,CGPA,Academic_Performance,Internship_Experience,Extra_Curricular_Score,Communication_Skills,Projects_Completed,Placement
0,107,6.61,6.28,8,0,8,8,4,0
1,97,5.52,5.37,8,0,7,8,0,0
2,109,5.36,5.83,9,0,3,1,1,0
3,122,5.47,5.75,6,1,1,6,1,0
4,96,7.91,7.69,7,0,8,10,2,0


In [12]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=["Placement"])
y=df["Placement"]
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42,stratify=y)
print(X_train.shape, X_test.shape)


(8000, 8) (2000, 8)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report
model=LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

Accuracy: 0.9035
              precision    recall  f1-score   support

           0       0.92      0.97      0.94      1668
           1       0.78      0.59      0.67       332

    accuracy                           0.90      2000
   macro avg       0.85      0.78      0.81      2000
weighted avg       0.90      0.90      0.90      2000



In [14]:
model_balanced=LogisticRegression(max_iter=1000,class_weight="balanced")
model_balanced.fit(X_train,y_train)
y_pred_balanced=model_balanced.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred_balanced))
print(classification_report(y_test,y_pred_balanced))

Accuracy: 0.867
              precision    recall  f1-score   support

           0       0.97      0.87      0.92      1668
           1       0.57      0.86      0.68       332

    accuracy                           0.87      2000
   macro avg       0.77      0.86      0.80      2000
weighted avg       0.90      0.87      0.88      2000



In [15]:
probs = model_balanced.predict_proba(X_test)
print(probs[:10])

[[9.93711253e-01 6.28874681e-03]
 [9.98186350e-01 1.81365044e-03]
 [9.99833777e-01 1.66222969e-04]
 [9.80970515e-01 1.90294846e-02]
 [4.49811195e-01 5.50188805e-01]
 [7.65041646e-01 2.34958354e-01]
 [8.41908219e-01 1.58091781e-01]
 [3.79362878e-02 9.62063712e-01]
 [7.40918610e-01 2.59081390e-01]
 [9.90722608e-01 9.27739153e-03]]


In [16]:
print(y_test.values[:10])

[0 0 0 0 0 0 0 1 0 0]


In [17]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=42)
rf_model.fit(X_train,y_train)
y_pred_rf=rf_model.predict(X_test)
print("accuracy:",accuracy_score(y_test,y_pred_rf))
print(classification_report(y_test, y_pred_rf))

accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1668
           1       1.00      1.00      1.00       332

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [18]:
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False))

IQ                        0.286338
Communication_Skills      0.272530
CGPA                      0.175818
Projects_Completed        0.137817
Prev_Sem_Result           0.108781
Extra_Curricular_Score    0.008699
Academic_Performance      0.007812
Internship_Experience     0.002204
dtype: float64


In [19]:
print("Duplicate rows in full dataset:", df.duplicated().sum())

Duplicate rows in full dataset: 0


In [20]:
from sklearn.tree import DecisionTreeClassifier,export_text
tree_model=DecisionTreeClassifier(max_depth=3,random_state=42)
tree_model.fit(X_train,y_train)
print(export_text(tree_model,feature_names=list(X_train.columns)))

|--- Communication_Skills <= 7.50
|   |--- IQ <= 110.50
|   |   |--- class: 0
|   |--- IQ >  110.50
|   |   |--- CGPA <= 8.01
|   |   |   |--- class: 0
|   |   |--- CGPA >  8.01
|   |   |   |--- class: 1
|--- Communication_Skills >  7.50
|   |--- CGPA <= 8.01
|   |   |--- IQ <= 110.50
|   |   |   |--- class: 0
|   |   |--- IQ >  110.50
|   |   |   |--- class: 1
|   |--- CGPA >  8.01
|   |   |--- Projects_Completed <= 1.50
|   |   |   |--- class: 0
|   |   |--- Projects_Completed >  1.50
|   |   |   |--- class: 1



In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=3, random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1668
           1       1.00      1.00      1.00       332

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [22]:
train_acc = rf_model.score(X_train, y_train)
test_acc = rf_model.score(X_test, y_test)
print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

Train accuracy: 1.0
Test accuracy: 1.0


In [23]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)
print("Train:", full_tree.score(X_train, y_train))
print("Test:", full_tree.score(X_test, y_test))

Train: 1.0
Test: 1.0


In [24]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_model, X, y, cv=5, scoring="accuracy")
print("Scores for each fold:", scores)
print("Mean:", scores.mean())
print("Std deviation:", scores.std())

Scores for each fold: [1. 1. 1. 1. 1.]
Mean: 1.0
Std deviation: 0.0


In [25]:
feature_ranges = X_train.describe().loc[["min", "max"]]
print(feature_ranges)

        IQ  Prev_Sem_Result   CGPA  Academic_Performance  \
min   41.0              5.0   4.54                   1.0   
max  158.0             10.0  10.46                  10.0   

     Internship_Experience  Extra_Curricular_Score  Communication_Skills  \
min                    0.0                     0.0                   1.0   
max                    1.0                    10.0                  10.0   

     Projects_Completed  
min                 0.0  
max                 5.0  


In [26]:
def check_profile_familiarity(user_profile: dict, feature_ranges) -> list:
    """
    user_profile: dict like {"IQ": 105, "CGPA": 8.2, ...}
    feature_ranges: the min/max table from Step 19

    Returns a list of warnings for any field outside the training range.
    """
    warnings = []
    for col in feature_ranges.columns:
        min_val = feature_ranges.loc["min", col]
        max_val = feature_ranges.loc["max", col]
        user_val = user_profile.get(col)

        if user_val is not None and (user_val < min_val or user_val > max_val):
            warnings.append(
                f"{col}={user_val} is outside the training range "
                f"({min_val}-{max_val}) — treat this prediction with caution"
            )
    return warnings

In [27]:
test_profile = {
    "IQ": 105, "Prev_Sem_Result": 8.0, "CGPA": 8.2, "Academic_Performance": 7,
    "Internship_Experience": 1, "Extra_Curricular_Score": 6,
    "Communication_Skills": 9, "Projects_Completed": 15   # deliberately unrealistic
}

result = check_profile_familiarity(test_profile, feature_ranges)
for w in result:
    print(w)

Projects_Completed=15 is outside the training range (0.0-5.0) — treat this prediction with caution


In [28]:
import joblib
joblib.dump(rf_model, "../models/placement_model.pkl")

['../models/placement_model.pkl']

In [29]:
X_train.columns

Index(['IQ', 'Prev_Sem_Result', 'CGPA', 'Academic_Performance',
       'Internship_Experience', 'Extra_Curricular_Score',
       'Communication_Skills', 'Projects_Completed'],
      dtype='str')

In [30]:
from sklearn.model_selection import GridSearchCV
param_grid={
    "n_estimators":[100,200,300],
    "max_depth":[None,5,10],
    "min_samples_split":[2,5,10]
}
grid_search=GridSearchCV(
    RandomForestClassifier(class_weight="balanced",random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)


Best parameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation score: 1.0


In [31]:
final_model = grid_search.best_estimator_
joblib.dump(final_model, "../models/placement_model.pkl")
print("Saved tuned model")

Saved tuned model
